Q1

In [24]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score
from sklearn.model_selection import train_test_split
%matplotlib inline

np.random.seed(1)

# Little boilerplate code to find out if we have a gpu
device = 'cpu'
if torch.cuda.device_count() > 0 and torch.cuda.is_available():
    print("Cuda installed! Running on GPU!")
    device = 'cuda'
else:
    print("No GPU available!")
print(f'Device: {device}')

Cuda installed! Running on GPU!
Device: cuda


Siesmic Event Detection

In [ ]:
# Loading data
Seismic_Data = np.load(r'D:\KAUST\semster2\ML_IN_GEO\SiesmicEventsDetectionPicking_Normalized.npz',allow_pickle=True) # Change the path to your local path
data = Seismic_Data['data']
label = Seismic_Data['label']
Pwave = Seismic_Data['P']
Swave = Seismic_Data['S']
Endcoda = Seismic_Data['E']
# Checking the label
k = 67
if label[k]==0:
    print('It is a noise waveform')
else:
    print('It is a seismic event waveform')
    
print('The data and label shapes are:',data.shape,label.shape)

It is a seismic event waveform
The data and label shapes are: (1000, 6000) (1000,)


Preparing Label Data for Detection (Sample-based)

In [26]:
Lab_Detection = np.zeros_like(data)
for i,j in enumerate(label):
    #Label for seismic event waveform
    if j==1:
        Lab_Detection[i,Pwave[i]:Endcoda[i]] =1
    #Label for noise waveform
    # Nothing to do since the label should be zero

Data preparation

In [27]:
from data_utils import create_dataloaders
train_loader, test_loader = create_dataloaders(
        data,
        Lab_Detection,
        batch_size=256,
        test_size=0.2,
        random_state=42,
        shuffle_train=True,
        transform=None)

Create Model Class

In [18]:
from model import MLP

input_dim = data.shape[1]
output_dim = data.shape[1]

models = {
    "Model1_ReLU": MLP(input_dim=input_dim, hidden_layers=[128], activation="relu", output_dim=output_dim),
    
    "Model2_ReLU": MLP(input_dim=input_dim, hidden_layers=[256, 128], activation="relu", output_dim=output_dim),
    
    "Model3_ReLU": MLP(input_dim=input_dim, hidden_layers=[512, 256, 128], activation="relu", output_dim=output_dim),
    
    "Model4_LeakyReLU": MLP(input_dim=input_dim, hidden_layers=[512, 256, 128], activation="leakyrelu", output_dim=output_dim),
    
    "Model5_GELU": MLP(input_dim=input_dim, hidden_layers=[512, 256, 128], activation="gelu", output_dim=output_dim),
}

Training and evaluating

In [31]:
from data_utils import train_model
import pandas as pd

results = []

for name, model in models.items():
    print(f"Training {name}")
    metrics = train_model(model, train_loader, test_loader, epochs=200)
    metrics["Model"] = name
    results.append(metrics)

results_df = pd.DataFrame(results)
print(results_df)

Training Model1_ReLU
Epoch [1/200] - Loss: 0.683909
Epoch [2/200] - Loss: 0.626484
Epoch [3/200] - Loss: 0.532089
Epoch [4/200] - Loss: 0.427212
Epoch [5/200] - Loss: 0.331325
Epoch [6/200] - Loss: 0.294110
Epoch [7/200] - Loss: 0.239651
Epoch [8/200] - Loss: 0.213889
Epoch [9/200] - Loss: 0.194554
Epoch [10/200] - Loss: 0.173207
Epoch [11/200] - Loss: 0.151419
Epoch [12/200] - Loss: 0.129826
Epoch [13/200] - Loss: 0.119057
Epoch [14/200] - Loss: 0.111679
Epoch [15/200] - Loss: 0.101800
Epoch [16/200] - Loss: 0.096624
Epoch [17/200] - Loss: 0.084062
Epoch [18/200] - Loss: 0.084450
Epoch [19/200] - Loss: 0.077788
Epoch [20/200] - Loss: 0.077078
Epoch [21/200] - Loss: 0.075981
Epoch [22/200] - Loss: 0.055641
Epoch [23/200] - Loss: 0.053150
Epoch [24/200] - Loss: 0.055532
Epoch [25/200] - Loss: 0.053071
Epoch [26/200] - Loss: 0.049447
Epoch [27/200] - Loss: 0.051492
Epoch [28/200] - Loss: 0.052633
Epoch [29/200] - Loss: 0.050545
Epoch [30/200] - Loss: 0.048407
Epoch [31/200] - Loss: 0.039

Save models

In [ ]:
import os

save_dir = "saved_models"
os.makedirs(save_dir, exist_ok=True)

torch.save(models["Model1_ReLU"].state_dict(), f"{save_dir}/model1.pth")
torch.save(models["Model2_ReLU"].state_dict(), f"{save_dir}/model2.pth")
torch.save(models["Model3_ReLU"].state_dict(), f"{save_dir}/model3.pth")
torch.save(models["Model4_LeakyReLU"].state_dict(), f"{save_dir}/model4.pth")
torch.save(models["Model5_GELU"].state_dict(), f"{save_dir}/model5.pth")

Q2 Transform data

In [ ]:
from data_utils import wavelet_transform, fft_transform, stft_transform
results2 = {}


###################################### Wavelet  ################################
train_loader_wav, test_loader_wav = create_dataloaders(
        data,
        Lab_Detection,
        batch_size=256,
        test_size=0.2,
        random_state=42,
        shuffle_train=True,
        transform=wavelet_transform)

model_wav = MLP(
    input_dim=train_loader_wav.dataset.data.shape[1],
    hidden_layers=[512,256,128],
    activation="leakyrelu",
    output_dim=output_dim
)

results2["Wavelet"] = train_model(model_wav, train_loader_wav, test_loader_wav, epochs=200)


####################################### FFT  ################################
train_loader_fft, test_loader_fft = create_dataloaders(
        data,
        Lab_Detection,
        batch_size=256,
        test_size=0.2,
        random_state=42,
        shuffle_train=True,
        transform=fft_transform)

model_fft = MLP(
    input_dim=train_loader_fft.dataset.data.shape[1],
    hidden_layers=[512,256,128],
    activation="leakyrelu",
    output_dim=output_dim
)

results2["FFT"] = train_model(
    model_fft,
    train_loader_fft,
    test_loader_fft,
    epochs=200,
)



###################################### STFT  ################################
train_loader_stft, test_loader_stft = create_dataloaders(
        data,
        Lab_Detection,
        batch_size=256,
        test_size=0.2,
        random_state=42,
        shuffle_train=True,
        transform=stft_transform)

model_stft = MLP(
    input_dim=train_loader_stft.dataset.data.shape[1],
    hidden_layers=[512,256,128],
    activation="leakyrelu",
    output_dim=output_dim
)

results2["STFT"] = train_model(
    model_stft,
    train_loader_stft,
    test_loader_stft,
    epochs=200
)


Epoch [1/200] - Loss: 0.685338
Epoch [2/200] - Loss: 0.588509
Epoch [3/200] - Loss: 0.395770
Epoch [4/200] - Loss: 0.301925
Epoch [5/200] - Loss: 0.234181
Epoch [6/200] - Loss: 0.234640
Epoch [7/200] - Loss: 0.205286
Epoch [8/200] - Loss: 0.205346
Epoch [9/200] - Loss: 0.208097
Epoch [10/200] - Loss: 0.210529
Epoch [11/200] - Loss: 0.179459
Epoch [12/200] - Loss: 0.179732
Epoch [13/200] - Loss: 0.156938
Epoch [14/200] - Loss: 0.146678
Epoch [15/200] - Loss: 0.118548
Epoch [16/200] - Loss: 0.097058
Epoch [17/200] - Loss: 0.084087
Epoch [18/200] - Loss: 0.069105
Epoch [19/200] - Loss: 0.063824
Epoch [20/200] - Loss: 0.067024
Epoch [21/200] - Loss: 0.063159
Epoch [22/200] - Loss: 0.072762
Epoch [23/200] - Loss: 0.065452
Epoch [24/200] - Loss: 0.056013
Epoch [25/200] - Loss: 0.065815
Epoch [26/200] - Loss: 0.055132
Epoch [27/200] - Loss: 0.053058
Epoch [28/200] - Loss: 0.051317
Epoch [29/200] - Loss: 0.046915
Epoch [30/200] - Loss: 0.046864
Epoch [31/200] - Loss: 0.063536
Epoch [32/200] - 

d:\anaconda3\envs\torch_env\Lib\site-packages\torch\functional.py:704: UserWarning: A window was not provided. A rectangular window will be applied,which is known to cause spectral leakage. Other windows such as torch.hann_window or torch.hamming_window can are recommended to reduce spectral leakage.To suppress this warning and use a rectangular window, explicitly set `window=torch.ones(n_fft, device=<device>)`. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\SpectralOps.cpp:842.)
  return _VF.stft(  # type: ignore[attr-defined]


Epoch [1/200] - Loss: 1.034602
Epoch [2/200] - Loss: 0.744469
Epoch [3/200] - Loss: 0.689092
Epoch [4/200] - Loss: 0.639952
Epoch [5/200] - Loss: 0.560176
Epoch [6/200] - Loss: 0.466734
Epoch [7/200] - Loss: 0.394465
Epoch [8/200] - Loss: 0.345231
Epoch [9/200] - Loss: 0.289659
Epoch [10/200] - Loss: 0.325942
Epoch [11/200] - Loss: 0.311659
Epoch [12/200] - Loss: 0.254541
Epoch [13/200] - Loss: 0.238307
Epoch [14/200] - Loss: 0.222847
Epoch [15/200] - Loss: 0.212910
Epoch [16/200] - Loss: 0.198916
Epoch [17/200] - Loss: 0.189211
Epoch [18/200] - Loss: 0.163203
Epoch [19/200] - Loss: 0.175575
Epoch [20/200] - Loss: 0.144350
Epoch [21/200] - Loss: 0.135215
Epoch [22/200] - Loss: 0.118019
Epoch [23/200] - Loss: 0.137623
Epoch [24/200] - Loss: 0.111337
Epoch [25/200] - Loss: 0.104128
Epoch [26/200] - Loss: 0.102831
Epoch [27/200] - Loss: 0.110332
Epoch [28/200] - Loss: 0.111087
Epoch [29/200] - Loss: 0.092767
Epoch [30/200] - Loss: 0.097805
Epoch [31/200] - Loss: 0.085660
Epoch [32/200] - 

In [21]:
###################################### ADD Spectrogram (optional)  ################################
import importlib
import data_utils  
importlib.reload(data_utils)
from data_utils import spectrogram_transform
from data_utils import train_model

train_loader_spec, test_loader_spec = create_dataloaders(
        data,
        Lab_Detection,
        batch_size=256,
        test_size=0.2,
        random_state=42,
        shuffle_train=True,
        transform=spectrogram_transform)

model_spec = MLP(
    input_dim=train_loader_spec.dataset.data.shape[1],
    hidden_layers=[512,256,128],
    activation="leakyrelu",
    output_dim=output_dim
)

results2["spectrogram"] = train_model(
    model_spec,
    train_loader_spec,
    test_loader_spec,
    epochs=200
)

Epoch [1/200] - Loss: 26.794855
Epoch [2/200] - Loss: 6.674899
Epoch [3/200] - Loss: 1.610445
Epoch [4/200] - Loss: 1.344552
Epoch [5/200] - Loss: 0.900903
Epoch [6/200] - Loss: 0.555213
Epoch [7/200] - Loss: 0.439191
Epoch [8/200] - Loss: 0.382076
Epoch [9/200] - Loss: 0.337442
Epoch [10/200] - Loss: 0.344083
Epoch [11/200] - Loss: 0.352136
Epoch [12/200] - Loss: 0.320893
Epoch [13/200] - Loss: 0.311287
Epoch [14/200] - Loss: 0.299370
Epoch [15/200] - Loss: 0.275952
Epoch [16/200] - Loss: 0.254065
Epoch [17/200] - Loss: 0.276197
Epoch [18/200] - Loss: 0.268846
Epoch [19/200] - Loss: 0.255595
Epoch [20/200] - Loss: 0.243848
Epoch [21/200] - Loss: 0.218339
Epoch [22/200] - Loss: 0.215133
Epoch [23/200] - Loss: 0.202095
Epoch [24/200] - Loss: 0.182672
Epoch [25/200] - Loss: 0.167222
Epoch [26/200] - Loss: 0.164925
Epoch [27/200] - Loss: 0.153823
Epoch [28/200] - Loss: 0.148537
Epoch [29/200] - Loss: 0.156005
Epoch [30/200] - Loss: 0.142316
Epoch [31/200] - Loss: 0.127781
Epoch [32/200] -

In [23]:
results2

{'spectrogram': {'precision': 0.7266719361951232,
  'recall': 0.7402976399501108,
  'f1': 0.7334215080271962,
  'pr_auc': 0.7803572369006775}}

In [54]:
original_df = results_df[results_df["Model"] == "Model4_LeakyReLU"]

original_row = {
    "Precision": original_df["precision"].values[0],
    "Recall": original_df["recall"].values[0],
    "F1": original_df["f1"].values[0],
    "PR-AUC": original_df["pr_auc"].values[0]
}

final_df = pd.DataFrame({
    "Method": ["Original", "Wavelet", "FFT", "STFT"],
    "Precision": [
        original_row["Precision"],
        results2["Wavelet"]["precision"],
        results2["FFT"]["precision"],
        results2["STFT"]["precision"]
    ],
    "Recall": [
        original_row["Recall"],
        results2["Wavelet"]["recall"],
        results2["FFT"]["recall"],
        results2["STFT"]["recall"]
    ],
    "F1": [
        original_row["F1"],
        results2["Wavelet"]["f1"],
        results2["FFT"]["f1"],
        results2["STFT"]["f1"]
    ],
    "PR-AUC": [
        original_row["PR-AUC"],
        results2["Wavelet"]["pr_auc"],
        results2["FFT"]["pr_auc"],
        results2["STFT"]["pr_auc"]
    ]
})

print(final_df)

     Method  Precision    Recall        F1    PR-AUC
0  Original   0.358494  0.482345  0.411298  0.341651
1   Wavelet   0.362552  0.454791  0.403467  0.340402
2       FFT   0.626024  0.496658  0.553887  0.610177
3      STFT   0.809802  0.879630  0.843273  0.912385


Q3 Reduce the training data size

In [ ]:
import importlib
import data_utils  
importlib.reload(data_utils)

ratio=[0.75,0.5,0.25,0.1]
results3 = {}
for r in ratio:
        train_loader, test_loader = create_dataloaders(
                data,
                Lab_Detection,
                batch_size=256,
                test_size=0.2,
                random_state=42,
                shuffle_train=True,
                transform=None,
                Reduce_the_size=r)
        
        model_reduce=MLP(input_dim=input_dim, hidden_layers=[512, 256, 128], activation="leakyrelu", output_dim=output_dim)
        results3[str(r)] = train_model(
                model_reduce,
                train_loader,
                test_loader,
                epochs=200
                )

Epoch [1/200] - Loss: 0.689774
Epoch [2/200] - Loss: 0.641282
Epoch [3/200] - Loss: 0.501464
Epoch [4/200] - Loss: 0.361760
Epoch [5/200] - Loss: 0.278046
Epoch [6/200] - Loss: 0.249044
Epoch [7/200] - Loss: 0.215956
Epoch [8/200] - Loss: 0.208196
Epoch [9/200] - Loss: 0.198232
Epoch [10/200] - Loss: 0.191612
Epoch [11/200] - Loss: 0.174045
Epoch [12/200] - Loss: 0.166158
Epoch [13/200] - Loss: 0.147598
Epoch [14/200] - Loss: 0.127883
Epoch [15/200] - Loss: 0.111688
Epoch [16/200] - Loss: 0.092329
Epoch [17/200] - Loss: 0.089052
Epoch [18/200] - Loss: 0.080122
Epoch [19/200] - Loss: 0.067012
Epoch [20/200] - Loss: 0.062670
Epoch [21/200] - Loss: 0.056582
Epoch [22/200] - Loss: 0.049220
Epoch [23/200] - Loss: 0.049466
Epoch [24/200] - Loss: 0.048291
Epoch [25/200] - Loss: 0.045257
Epoch [26/200] - Loss: 0.045150
Epoch [27/200] - Loss: 0.044449
Epoch [28/200] - Loss: 0.043098
Epoch [29/200] - Loss: 0.040746
Epoch [30/200] - Loss: 0.035188
Epoch [31/200] - Loss: 0.035743
Epoch [32/200] - 

In [66]:
results3_df = pd.DataFrame(results3).T
results3_df

,precision,recall,f1,pr_auc
0.75,0.363244,0.494381,0.418787,0.337951
0.5,0.364763,0.458333,0.406229,0.324689
0.25,0.362070,0.488678,0.415953,0.322178
0.1,0.389761,0.472761,0.427268,0.359045
